# ڕاهێنانی مۆدێلی پاشماوە لەسەر وێنەکانی سلێمانی
*Fine-tune the litter model on our own Slemani photos (free Colab GPU)*

ئەم دەفتەرە مۆدێلی گشتیی پاشماوە لەسەر ئەو وێنانەی سلێمانی ڕادەهێنێت کە خۆمان نیشانەمان کردوون. پاشان لەسەر
ئەو وێنانەی تاقیکردنەوە کە هەرگیز بۆ ڕاهێنان بەکارنەهاتوون، لەگەڵ مۆدێلی گشتی بەراوردی دەکات. لە کۆتاییدا
`slemani.pt` دادەبەزێنیت.

خانەکان یەک لە دوای یەک کار پێبکە (Shift+Enter). هەمان فەرمانەکانی `backend/tools/` بەکاردەهێنێت، کەواتە
لەسەر Kaggle یان هەر کۆمپیوتەرێکی خاوەن GPU هەمان هەنگاوەکان کار دەکەن.

## ١. GPU بپشکنە
*Check the GPU*

لە مێنیوی **Runtime ← Change runtime type** دا **T4 GPU** هەڵبژێرە. ئەگەر خانەی خوارەوە `NONE` بنووسێت،
ڕاهێنان زۆر هێواش دەبێت.

In [ ]:
!nvidia-smi -L
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

## ٢. پڕۆژەکە و ultralytics
*Clone the repo and install ultralytics*

ئەگەر ڕیپۆکە تایبەت (private) بێت، تۆکنێکی GitHub بخە ناو بەستەرەکە
(`https://<token>@github.com/...`) و تۆکنەکە لە دەفتەرەکەدا پاشەکەوت مەکە.

In [ ]:
REPO = "https://github.com/randzana/Slemani-GreenLegacy"
BRANCH = ""   # empty = the default branch

import os, shutil, sys, zipfile
from pathlib import Path

if not os.path.isdir("/content/Slemani-GreenLegacy"):
    branch = f"-b {BRANCH}" if BRANCH else ""
    !git clone --depth 1 {branch} {REPO} /content/Slemani-GreenLegacy
%cd /content/Slemani-GreenLegacy/backend
sys.path.insert(0, os.getcwd())
# psycopg and flask: tools/evaluate.py imports the app package, which imports them
!pip install -q ultralytics "psycopg[binary]" flask

## ٣. وێنە نیشانەکراوەکان
*Bring the annotation export (a YOLO-format zip)*

لە Label Studio، CVAT یان Roboflow بە شێوازی YOLO هەناردەی بکە (export). zip ەکە دەبێت ئەمانەی تێدابێت:

- `images/` — وێنەکان
- `labels/` — بۆ هەر وێنەیەک فایلێکی `.txt` بە هەمان ناو (وێنەی پاک: بێ فایل یان فایلی بەتاڵ)
- `classes.txt` — ناوی هەر پۆلێک لە دێڕێکدا

یان zip ەکە بخە ناو Google Drive (`USE_DRIVE = True` و ڕێڕەوەکەی بگۆڕە)، یان `USE_DRIVE = False` بکە و
ڕاستەوخۆ لێرە باری بکە (upload).

In [ ]:
USE_DRIVE = True
ZIP_IN_DRIVE = "/content/drive/MyDrive/GreenLegacy/slemani_export.zip"   # change to where your zip is

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = ZIP_IN_DRIVE
else:
    from google.colab import files
    name = next(iter(files.upload()))
    zip_path = shutil.move(name, f"/content/{name}")

shutil.rmtree("/content/export", ignore_errors=True)
zipfile.ZipFile(zip_path).extractall("/content/export")
# The export may sit inside one more folder: use the folder that holds classes.txt.
found = sorted(Path("/content/export").rglob("classes.txt"))
assert found, "no classes.txt in the zip: export in YOLO format (images/, labels/, classes.txt)"
CLASSES = found[0]
IMAGES, LABELS = CLASSES.parent / "images", CLASSES.parent / "labels"

from tools.train import label_kind
KIND = label_kind(LABELS)
print(len(list(IMAGES.iterdir())), "images,", len(list(LABELS.glob("*.txt"))), "label files,", KIND)
print("classes:", CLASSES.read_text(encoding="utf-8").split())

## ٤. مۆدێلی گشتیی پاشماوە
*Download the general trash model (Hugging Face works on Colab)*

`turhancan97/yolov8-segment-trash-detection` (YOLOv8 segmentation، مۆڵەتی MIT). ناوی ئەم مۆدێلە و
TACO (CC BY 4.0) لە سلایدەکاندا بنووسە.

In [ ]:
!python tools/download_model.py --trash
from tools.download_model import TRASH_REPO, trash_files
GENERAL = f"models/{Path(trash_files(TRASH_REPO)[0]).name}"   # the file the script just saved
print("general trash model:", GENERAL)

## ٥. دابەشکردنی وێنەکان
*Split into train / val / held-out test*

`TEST` وێنە (بنەڕەت ٥٠) لە ڕاهێنان دوور دەخرێنەوە و تەنها بۆ هەڵسەنگاندنی کۆتایین. بەشێک لە وێنە پاکەکانیش
دەچنە ناویانەوە، بۆ ئەوەی ببینین مۆدێلەکە لە وێنەی پاکدا پاشماوە دەدۆزێتەوە یان نا، چونکە ئەوە خاڵی بەخۆڕایی
دەدات. بۆ دابەشکردنێکی نوێ، سەرەتا `!rm -rf datasets/slemani` لێبدە.

In [ ]:
TEST = 50   # held-out photos, never trained on
!python tools/train.py split --images "{IMAGES}" --labels "{LABELS}" --classes "{CLASSES}" --out datasets/slemani --test {TEST}

## ٦. ڕاهێنان
*Train: the best weights become models/slemani.pt*

- لەیبڵی polygon: مۆدێلی پاشماوەکە (segmentation) دەبێتە بنەما.
- لەیبڵی چوارگۆشە (box): **`yolov8s.pt`** بەکاربهێنە، چونکە مۆدێلی segmentation لە box فێرنابێت.
  خانەی خوارەوە خۆی هەڵیدەبژێرێت.

لە کۆتاییدا `train.py` ئەنجامی mAP لەسەر وێنەکانی تاقیکردنەوە دەنووسێت.

In [ ]:
if KIND == "boxes":
    !python tools/download_model.py --stock yolov8s.pt
    BASE = "models/yolov8s.pt"
else:
    BASE = GENERAL
print("base model:", BASE)
!python tools/train.py train --data datasets/slemani/data.yaml --base {BASE} --epochs 60 --imgsz 640 --batch 16 --name slemani

## ٧. بەراورد: مۆدێلی گشتی بەرامبەر مۆدێلی سلێمانی
*Compare the general and the fine-tuned model on the held-out photos*

ئەگەر `slemani.pt` باشتر نەبوو (لە دۆزینەوەی پاشماوە لە وێنەی پیس، و نەدۆزینەوەی لە وێنەی پاک)، مۆدێلی
گشتی بهێڵەوە. خشتەکە بۆ سلایدەکانە: تەنها ئەم ژمارانە بەکاربهێنە، چونکە لە وێنەی ڕاستەقینەی سلێمانی هاتوون.

In [ ]:
!python tools/evaluate.py datasets/slemani/test --model {GENERAL} --model models/slemani.pt --csv datasets/slemani/compare.csv

## ٨. پلەکانی پیسی ١-٥ (ئارەزوومەندانە)
*Optional: tune the dirtiness bands from levels given by hand*

١. `labels.csv` دابەزێنە و ستوونی `level` بۆ هەر وێنەیەک پڕبکەرەوە (١ تا ٥، وێنەی پاک ٠).

٢. فایلە پڕکراوەکە لە خانەی دووەمدا باربکەرەوە. `--suggest-bands` سنوورەکانی `LEVEL_COUNT_BANDS` پێشنیار دەکات.

In [ ]:
from google.colab import files
files.download("datasets/slemani/test/labels.csv")

In [ ]:
from google.colab import files
name = next(iter(files.upload()))   # the labels.csv with the level column filled in
shutil.move(name, "datasets/slemani/test/labels.csv")
!python tools/evaluate.py datasets/slemani/test --model models/slemani.pt --suggest-bands

## ٩. مۆدێلەکە دابەزێنە
*Download models/slemani.pt*

لەسەر لاپتۆپەکە بیخە ناو `backend/models/` و سێرڤەرەکە بەم شێوەیە دەستپێبکە:

```
MODEL_PATH=models/slemani.pt LITTER_CLASSES=* python run.py
```

ئەگەر هەنگاوی ٨ت کرد، `LEVEL_COUNT_BANDS=...`یش زیاد بکە. فایلەکانی `.pt` جۆری pickle ن: تەنها ئەوانە
بەکاربهێنە کە سەرچاوەکەیان متمانەپێکراوە.

In [ ]:
from google.colab import files
files.download("models/slemani.pt")
if USE_DRIVE:   # also keep the model and the training curves next to the zip in Drive
    keep = Path(ZIP_IN_DRIVE).parent
    shutil.copy("models/slemani.pt", keep / "slemani.pt")
    shutil.copytree("runs/slemani", keep / "runs_slemani", dirs_exist_ok=True)
    print("copied to", keep)